# Cross Validation

In [1]:
import numpy as np
import pandas as pd
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 1. Load Data
X = np.load('../data/processed/X_train_features.npy')
y = np.load('../data/processed/y_train.npy')
classes = ['Smoke', 'Fire', 'Non_Fire']

# 2. Initialize StratifiedKFold (Requirement: K=3)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

fold_results = []

print(f"Starting {skf.get_n_splits()}-fold Stratified Cross-Validation...")

# 3. CV Loop
for i, (train_index, val_index) in enumerate(skf.split(X, y)):
    print(f"\n--- Processing Fold {i+1} ---")
    
    # Split data for this fold
    X_train_fold, X_val_fold = X[train_index], X[val_index]
    y_train_fold, y_val_fold = y[train_index], y[val_index]
    
    # Optional: Feature Scaling (Recommended for SVM/Logistic)
    scaler = StandardScaler()
    X_train_fold = scaler.fit_transform(X_train_fold)
    X_val_fold = scaler.transform(X_val_fold)
    
    # Define our best model (Stacking Ensemble as defined in issue #10)
    # Re-using the logic from 02a_modeling
    base_models = [
        ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
        ('xgb', XGBClassifier(n_estimators=100, random_state=42)),
        ('svm', SVC(kernel='linear', probability=True, random_state=42))
    ]
    clf = StackingClassifier(estimators=base_models, final_estimator=LogisticRegression())
    
    # Train
    clf.fit(X_train_fold, y_train_fold)
    
    # Predict & Evaluate
    y_pred = clf.predict(X_val_fold)
    acc = accuracy_score(y_val_fold, y_pred)
    f1 = f1_score(y_val_fold, y_pred, average='macro')
    
    fold_results.append({'fold': i+1, 'accuracy': acc, 'f1_macro': f1})
    print(f"Fold {i+1} Results -> Accuracy: {acc:.4f}, F1-Macro: {f1:.4f}")

# 4. Aggregate Results (Requirement: Mean and STD)
df_results = pd.DataFrame(fold_results)
mean_f1 = df_results['f1_macro'].mean()
std_f1 = df_results['f1_macro'].std()

print("\n--- Final CV Summary ---")
print(f"Mean F1-Macro: {mean_f1:.4f}")
print(f"Standard Deviation: {std_f1:.4f}")

# Check Acceptance Criteria: STD < 0.03
if std_f1 < 0.03:
    print("✅ Success: Model is stable (STD < 0.03)")
else:
    print("⚠️ Warning: Model variance is high (STD >= 0.03)")

# 5. Export Results to JSON (Requirement: reports/02_cv_results.json)
cv_summary = {
    "mean_f1_macro": float(mean_f1),
    "std_f1_macro": float(std_f1),
    "fold_details": fold_results
}

os.makedirs('../reports', exist_ok=True)
with open('../reports/02_cv_results.json', 'w') as f:
    json.dump(cv_summary, f, indent=4)

Starting 3-fold Stratified Cross-Validation...

--- Processing Fold 1 ---
Fold 1 Results -> Accuracy: 0.9833, F1-Macro: 0.9833

--- Processing Fold 2 ---
Fold 2 Results -> Accuracy: 0.9839, F1-Macro: 0.9839

--- Processing Fold 3 ---
Fold 3 Results -> Accuracy: 0.9826, F1-Macro: 0.9825

--- Final CV Summary ---
Mean F1-Macro: 0.9832
Standard Deviation: 0.0007
✅ Success: Model is stable (STD < 0.03)
